In [1]:
import pandas as pd

In [7]:
out_df = pd.read_parquet("https://s3.us-west-2.amazonaws.com/pudl.catalyst.coop/nightly/out_sec10k__parents_and_subsidiaries.parquet")

There are duplicate records. Oops. This is an easy fix.

In [14]:
out_df = out_df.drop_duplicates()

`company_id_sec10k` is an ID assigned by us. `central_index_key` (CIK) is an ID assigned by the SEC. When a company is an SEC 10-K filer themselves then their `company_id_sec10k` is the same as the `central_index_key`. When a company is an Ex. 21 subsidiary that doesn't file their own 10-K, then their `company_id_sec10k` is a concatenation of their parent company's CIK, the subsidiary name, and the location of incorporation. This is to uniquely identify subsidiaries who don't have a CIK.

In [22]:
out_df[out_df.files_sec10k].head(3)

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
0,0000001800,edgar/data/1800/0000912057-94-000771.txt,1994-03-03,0000001800,NaN,one abbott park rd,None,abbott park,il,abbott laboratories,None,None,None,None,il,illinois,360698440,True,0000001800,NaN
1,0000883702,edgar/data/883702/0000912057-94-002192.txt,1994-06-29,0000883702,NaN,13500 south perry ave,None,riverdale,il,acme metals inc /de/,None,None,None,None,de,delaware,363802419,True,None,NaN
2,0000061478,edgar/data/61478/0000912057-94-004277.txt,1994-12-22,0000061478,NaN,4900 west 78th st,None,minneapolis,mn,adc telecommunications inc,1985-06-05,magnetic controls co,telephone & telegraph apparatus,3661,mn,minnesota,410743912,True,0000061478,NaN


In [21]:
out_df[~out_df.files_sec10k].head(3)

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
514,0000001750_aar allen airmotive incorporated_il...,edgar/data/1750/0000912057-94-002818.txt,1994-08-24,None,NaN,None,None,None,None,"aar allen airmotive, inc",None,None,None,None,None,illinois,None,False,0000001750,NaN
515,0000001750_aar aviation services incorporated_...,edgar/data/1750/0000912057-94-002818.txt,1994-08-24,None,NaN,None,None,None,None,"aar aviation services, inc",None,None,None,None,None,new york,None,False,0000001750,NaN
516,0000001750_aar aviation trading incorporated_i...,edgar/data/1750/0000912057-94-002818.txt,1994-08-24,None,NaN,None,None,None,None,"aar aviation trading, inc",None,None,None,None,None,illinois,None,False,0000001750,NaN


To trace to a parent company's information, one must conduct a join on `parent_company_central_index_key`.

In [29]:
out_df[out_df.company_name_raw.str.contains("nextera")].head(2)[["company_id_sec10k", "company_name_raw", "parent_company_central_index_key"]]

,company_id_sec10k,company_name_raw,parent_company_central_index_key
162172,0001070534_nextera business performance soluti...,"nextera business performance solutions group, inc",0001070534
181812,0001070534,nextera enterprises inc,None


We used record linkage to add a `utility_id_eia` onto companies that report to the EIA.

In [30]:
out_df.utility_id_eia.isnull().value_counts()

utility_id_eia
True     1106722
False       6244
Name: count, dtype: int64

Company ID, report date, and parent company ID should be a primary key, but often the ownership percentage is null for one record and filled in for the other. We should unify these discrepancies.

In [31]:
out_df[out_df[["company_id_sec10k", "report_date", "parent_company_central_index_key"]].duplicated(
    keep=False
)].sort_values(
    by=["company_id_sec10k", "report_date"]
).head(4)[[]]

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
574839,0000005272,edgar/data/5272/0001193125-12-143882.txt,2012-03-30,0000005272,NaN,70 pne st,None,new york,ny,american international group inc,1970-05-07,american international enterprises inc,"fire, marine & casualty insurance",6331,de,delaware,132592361,True,0000005272,0.02
575024,0000005272,edgar/data/5272/0001193125-12-143882.txt,2012-03-30,0000005272,NaN,70 pne st,None,new york,ny,american international group inc,1970-05-07,american international enterprises inc,"fire, marine & casualty insurance",6331,de,delaware,132592361,True,0000005272,NaN
645771,0000005272,edgar/data/5272/0001047469-14-001096.txt,2014-02-20,0000005272,NaN,180 maiden ln,None,new york,ny,american international group inc,1970-05-07,american international enterprises inc,"fire, marine & casualty insurance",6331,de,delaware,132592361,True,0000005272,0.02
645795,0000005272,edgar/data/5272/0001047469-14-001096.txt,2014-02-20,0000005272,NaN,180 maiden ln,None,new york,ny,american international group inc,1970-05-07,american international enterprises inc,"fire, marine & casualty insurance",6331,de,delaware,132592361,True,0000005272,NaN


This output table doesn't have a complete time series of SEC records and ownership changes. It only has the most recent record for each unique company ID and street address (so there's a new record every time it changes address). This is largely due to how I needed the table to look for record linkage for EIA. For the complete time series you need the core table. But the core table doesn't have a connection to EIA. Let's look at Berkshire Hathaway.

In [54]:
out_df[out_df.company_id_sec10k == "0001067983"]

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
550962,0001067983,edgar/data/1067983/0001193125-11-048914.txt,2011-02-28,0001067983,NaN,1440 kiewit plz,None,omaha,ne,berkshire hathaway inc,1998-08-10,nbh inc,"fire, marine & casualty insurance",6331,de,delaware,470813844,True,None,NaN
962002,0001067983,edgar/data/1067983/0000950170-23-004451.txt,2023-02-27,0001067983,NaN,3555 farnam st,None,omaha,ne,berkshire hathaway inc,1998-08-10,nbh inc,"fire, marine & casualty insurance",6331,de,delaware,470813844,True,None,NaN


Look at the subsidiaries of Berkshire Hathaway

In [57]:
out_df[out_df.parent_company_central_index_key == "0001067983"]

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
23005,0000079958,edgar/data/79958/0000079958-96-000010.txt,1996-07-01,0000079958,NaN,4600 se harney dr,None,portland,or,precision castparts corp,None,None,iron & steel foundries,3320,or,oregon,930460598,True,0001067983,NaN
25838,0000816512,edgar/data/816512/0000918695-96-000029.txt,1996-06-25,0000816512,NaN,4726 airport hwy,None,louisville,tn,vanderbilt mortgage & finance inc,None,None,asset-backed securities,6189,tn,tennessee,620997810,True,0001067983,NaN
33533,0000037481,edgar/data/37481/0000037481-96-000005.txt,1996-03-29,0000037481,NaN,laguardia airport,marine air terminal,flushing,ny,flightsafety international inc,None,None,services-educational services,8200,ny,new york,111671001,True,0001067983,NaN
33859,0000277795,edgar/data/277795/0000277795-96-000001.txt,1996-03-25,0000277795,NaN,geico plz,None,washington,dc,geico corp,None,None,"fire, marine & casualty insurance",6331,de,delaware,521135801,True,0001067983,NaN
45055,0000934612,edgar/data/934612/0000950131-97-002298.txt,1997-03-31,0000934612,NaN,3800 continental plz,777 main st,ft worth,tx,burlington northern santa fe corp,1995-09-13,burlington northern sante fe corp,"railroads, line-haul operating",4011,de,delaware,411804964,True,0001067983,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
975253,0001067983_xtra corporation_delaware,edgar/data/1067983/0000950170-23-004451.txt,2023-02-27,None,NaN,None,None,None,None,xtra corporation,None,None,None,None,None,delaware,None,False,0001067983,NaN
975255,0001067983_xtra finance corporation_delaware,edgar/data/1067983/0000950170-23-004451.txt,2023-02-27,None,NaN,None,None,None,None,xtra finance corporation,None,None,None,None,None,delaware,None,False,0001067983,NaN
975264,0001067983_xtra lease limited liability compan...,edgar/data/1067983/0000950170-23-004451.txt,2023-02-27,None,NaN,None,None,None,None,xtra lease llc,None,None,None,None,None,delaware,None,False,0001067983,NaN
975266,0001067983_xtra limited liability company_maine,edgar/data/1067983/0000950170-23-004451.txt,2023-02-27,None,NaN,None,None,None,None,xtra llc,None,None,None,None,None,maine,None,False,0001067983,NaN


Look at who owns Georgia Power

In [59]:
out_df[out_df.company_name_raw.str.contains("georgia power")].head(5)

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
265574,0000041091_georgia power capital trust iv_dela...,edgar/data/41091/0000092122-04-000119.txt,2004-03-01,None,NaN,None,None,None,None,georgia power capital trust iv,None,None,None,None,None,delaware,None,False,0000041091,NaN
267634,0000044545_georgia power capital trust iv_dela...,edgar/data/44545/0000092122-04-000119.txt,2004-03-01,None,NaN,None,None,None,None,georgia power capital trust iv,None,None,None,None,None,delaware,None,False,0000044545,NaN
299562,0000041091,edgar/data/44545/0000950144-05-001905.txt,2005-03-01,0000041091,7140.0,241 ralph mcgill blvd,ne bin 10116,atlanta,ga,georgia power co,None,None,electric services,4911,ga,georgia,580257110,True,0000003153,NaN
300145,0000041091,edgar/data/44545/0000950144-05-001905.txt,2005-03-01,0000041091,7140.0,241 ralph mcgill blvd,ne bin 10116,atlanta,ga,georgia power co,None,None,electric services,4911,ga,georgia,580257110,True,0000041091,NaN
300153,0000041091,edgar/data/44545/0000950144-05-001905.txt,2005-03-01,0000041091,7140.0,241 ralph mcgill blvd,ne bin 10116,atlanta,ga,georgia power co,None,None,electric services,4911,ga,georgia,580257110,True,0000044545,NaN


Due to weirdness with the Ex. 21 extraction model, Georgia Power has a bunch of different parent companies for the 2005 record.

In [62]:
out_df[out_df.company_id_sec10k == "0000041091"][["company_name_raw", "report_date", "street_address", "parent_company_central_index_key"]]

,company_name_raw,report_date,street_address,parent_company_central_index_key
299562,georgia power co,2005-03-01,241 ralph mcgill blvd,0000003153
300145,georgia power co,2005-03-01,241 ralph mcgill blvd,0000041091
300153,georgia power co,2005-03-01,241 ralph mcgill blvd,0000044545
306085,georgia power co,2005-03-01,241 ralph mcgill blvd,0000066904
307851,georgia power co,2005-03-01,241 ralph mcgill blvd,0000086940
307904,georgia power co,2005-03-01,241 ralph mcgill blvd,0000092122
308115,georgia power co,2005-03-01,241 ralph mcgill blvd,0001004155
309534,georgia power co,2005-03-01,241 ralph mcgill blvd,0001160661


One of these parent companies is Southern Company, but we can't trace when this ownership change happened without the full time series.

In [63]:
out_df[out_df.company_id_sec10k == "0000092122"]

,company_id_sec10k,filename_sec10k,report_date,central_index_key,utility_id_eia,street_address,address_2,city,state,company_name_raw,name_change_date,company_name_former,industry_description_sic,industry_id_sic,state_of_incorporation,location_of_incorporation,company_id_irs,files_sec10k,parent_company_central_index_key,fraction_owned
54712,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000003153,NaN
54726,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000041091,NaN
54727,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000044545,NaN
54738,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000066904,NaN
54741,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000086940,NaN
54742,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000092122,NaN
54743,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0001004155,NaN
54744,0000092122,edgar/data/41091/0000092122-97-000016.txt,1997-03-25,0000092122,NaN,64 perimeter ctr east,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0001160661,NaN
336579,0000092122,edgar/data/92122/0000092122-06-000182.txt,2006-03-17,0000092122,NaN,270 peachtree st,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000003153,NaN
336607,0000092122,edgar/data/92122/0000092122-06-000182.txt,2006-03-17,0000092122,NaN,270 peachtree st,p o box 2641,atlanta,ga,southern co,None,None,electric services,4911,de,delaware,580690070,True,0000041091,NaN
